# clock/10.1 — gaze heads vs concept-copier (dual-route induction) heads

Sibling of `10_goal_vs_FV_heads`, swapping the pair:
- **TARGET_SET = gaze heads** (our step-lookback heads) — cf. goal heads in nb10.
- **probe set = concept-copier heads** (Feucht et al., dual-route induction) — cf. FV heads in nb10.

**TODOs**
1. Load gaze (target) + concept-copier heads.
2. **Sanity check** the concept heads' functional purpose by replicating the dual-route demo
   (§2 attention / induction pattern): qualitative heatmap + a ~32-prompt smoke.
3. Plot **overlap** (concept vs gaze).
4. **Compare qualitatively** — *do concept heads show gaze behavior?* → the gaze attention
   visualization, but fed the concept-copier heads.

> ⚠️ **Blocker:** concept-copier head orderings only exist for the paper models + a 7B smoke; the
> **32B dual-route run hasn't finished**, so `head_orderings/Qwen2.5-Coder-32B-Instruct/concept_copying.json`
> is absent. Either finish that run (`job_list/qwen32b_dualroute_causal_localization.json`) or set
> `MODEL_NAME` to a model that has one (e.g. `meta-llama/Meta-Llama-3-8B`) to run §2–§4 today.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, json, glob, random, logging
sys.path.append("../..")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "3,4")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from rich.console import Console
from rich.theme import Theme
from rich.table import Table
from rich.markup import escape
from src.utils import logging_utils, env_utils
logging_utils.configure(level=logging.INFO)
logger = logging.getLogger(__name__)
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)

In [ ]:
from src.models import ModelandTokenizer
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(model_key=MODEL_NAME, dtype=torch.bfloat16,
                       device_map="auto", attn_implementation="eager")
N_LAYER = mt.n_layer
N_HEADS = mt.model.config.num_attention_heads
print(f"n_layer={N_LAYER}  n_embd={mt.n_embd}  n_heads={N_HEADS}")

## 1. Head sets — gaze (TARGET) + concept-copier heads

In [ ]:
from scripts.clock.score_condition_heads import load_head_scores

TOPK       = 64          # shared K for gaze & concept (fair overlap), like nb10's FV top-K
GAZE_QSPAN = 15

gaze_scores = load_head_scores(model=MODEL_NAME, version="conditions",
                               q_span=GAZE_QSPAN, compare_with="mean")["combined"]   # (L, H)
_flat = np.argsort(gaze_scores, axis=None)[::-1][:TOPK]
gaze_heads = [(int(i // N_HEADS), int(i % N_HEADS)) for i in _flat]
print(f"gaze heads (TARGET, top-{TOPK} step-lookback q_span={GAZE_QSPAN}): {len(gaze_heads)}")

In [ ]:
# concept-copier head ordering from the vendored dual-route run (ranked list [[layer, head], ...])
DUALROUTE = os.path.join(REPO := os.path.abspath("../.."), "existing_works", "dual-route-induction")
CONCEPT_TOPK = TOPK
cc_dir  = os.path.join(DUALROUTE, "cache", "head_orderings", MODEL_NAME.split("/")[-1])
cc_path = os.path.join(cc_dir, "concept_copying.json")
if not os.path.exists(cc_path):
    alts = sorted(glob.glob(os.path.join(cc_dir, "concept_copying*.json")))
    assert alts, (
        f"no concept-copier ordering under {cc_dir}.\n"
        f"  Finish the dual-route causal_scores run for {MODEL_NAME} "
        f"(job_list/qwen32b_dualroute_causal_localization.json),\n"
        f"  or set MODEL_NAME to a paper model that has one (e.g. meta-llama/Meta-Llama-3-8B).")
    cc_path = alts[-1]
    print(f"[warn] full concept_copying.json absent; using {os.path.basename(cc_path)} (partial/smoke)")
concept_ranked = json.load(open(cc_path))                       # ranked, best first
concept_heads  = [tuple(x) for x in concept_ranked[:CONCEPT_TOPK]]
print(f"concept-copier heads (top-{CONCEPT_TOPK} from {os.path.basename(cc_path)}): {len(concept_heads)}")

## 2. Concept-copier sanity check — replicate the dual-route demo (§2 attention / induction)

Feucht et al.'s demo (`head_set_vis`) shows that on a **repeated** prompt the concept/induction heads
attend from a token in the 2nd copy back into the 1st copy — the copy/induction signature. We
replicate that heatmap, then a small quantitative smoke (~32 prompts): the attention mass concept
heads place into the first copy, vs a random-head baseline.

In [ ]:
from src.attention import get_attention_matrices

# ---- qualitative: dual-route head_set_vis replica (attention from a 2nd-copy token) ----
DEMO_PROMPT = "By the false azure in the windowpane"      # the dual-route demo's sentence
FROMTOK     = -4
rep = f"{DEMO_PROMPT}\n{DEMO_PROMPT}"
ai  = get_attention_matrices(rep, mt, value_weighted=True)
toks = ai.tokenized_prompt
xlab = [t for t in toks[:FROMTOK + 1]]
K = min(8, len(concept_heads))
grid = np.stack([np.asarray(ai.attention_matrices[l, h, FROMTOK, :FROMTOK + 1]) for (l, h) in concept_heads[:K]])

fig, ax = plt.subplots(figsize=(min(0.32 * len(xlab) + 2, 16), 3.6))
im = ax.imshow(grid, aspect="auto", cmap="Purples")
ax.set_xticks(range(len(xlab))); ax.set_xticklabels(xlab, rotation=90, fontsize=7)
ax.set_yticks(range(K)); ax.set_yticklabels([f"L{l}H{h}" for l, h in concept_heads[:K]], fontsize=7)
ax.set_title(f"top-{K} concept-copier heads — value-weighted attention from tok {FROMTOK}")
fig.colorbar(im, ax=ax, fraction=0.025); plt.tight_layout(); plt.show()

# ---- quantitative smoke (~32 prompts): first-copy attention mass, concept vs random heads ----
subj = ["the cat", "a robin", "my friend", "the engineer", "her sister", "the old man", "a student", "the pilot"]
pred = ["walked to the river", "opened the wooden door", "counted the bright stars", "sold the red bicycle"]
SENTS = [f"{s} {p}" for s in subj for p in pred]           # 32 prompts

_rng = random.Random(0)
random_heads = [(_rng.randrange(N_LAYER), _rng.randrange(N_HEADS)) for _ in range(len(concept_heads))]

def first_copy_mass(heads):
    """mean over prompts & heads of attention mass from the last token into the 1st copy."""
    vals = []
    for s in SENTS:
        a = get_attention_matrices(f"{s}\n{s}", mt, value_weighted=True)
        tk = a.tokenized_prompt
        nl = next((i for i, t in enumerate(tk) if "\n" in t), len(tk) // 2)   # 1st copy = [0, nl)
        A = np.asarray(a.attention_matrices)                                   # (L, H, S, S)
        row = A[[l for l, _ in heads], [h for _, h in heads], -1, :]           # (n_heads, S)
        vals.append(row[:, :nl].sum(axis=1).mean())                            # mass into 1st copy
    return float(np.mean(vals))

cc_mass, rnd_mass = first_copy_mass(concept_heads), first_copy_mass(random_heads)
t = Table(title=f"induction smoke — first-copy attention mass ({len(SENTS)} repeated prompts)")
t.add_column("head set"); t.add_column("mean 1st-copy attn (last tok)", justify="right")
t.add_row(f"concept-copier (top-{len(concept_heads)})", f"{cc_mass:.3f}")
t.add_row(f"random ({len(random_heads)})", f"{rnd_mass:.3f}")
rich_print(t)
rich_print(f"[bold]concept / random ratio = {cc_mass / max(rnd_mass, 1e-9):.2f}×[/bold]  "
           "(>1 -> concept heads preferentially look back into the repeated copy = induction behavior)")

## 3. Location + overlap — concept-copier vs gaze  (mirrors nb10 §3a)

In [ ]:
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

gset, cset = set(gaze_heads), set(concept_heads)
inter, union = gset & cset, gset | cset
jacc = len(inter) / len(union) if union else 0.0
print(f"gaze: {len(gset)}  concept: {len(cset)}  intersection: {len(inter)}  union: {len(union)}  Jaccard: {jacc:.3f}")

grid = np.zeros((N_LAYER, N_HEADS))
for (l, h) in gaze_heads:    grid[l, h] += 1        # 1 = gaze
for (l, h) in concept_heads: grid[l, h] += 2        # 2 = concept, 3 = both
cmap = ListedColormap(["#f0f0f0", "#55a868", "#8172b3", "#c44e52"])
fig, ax = plt.subplots(figsize=(9, 5))
ax.imshow(grid.T, aspect="auto", cmap=cmap, vmin=0, vmax=3, origin="lower")
ax.set_xlabel("layer"); ax.set_ylabel("head"); ax.set_title("gaze vs concept-copier heads")
ax.legend(handles=[mpatches.Patch(color=c, label=l) for c, l in
                   [("#55a868", "gaze only"), ("#8172b3", "concept only"), ("#c44e52", "both")]], loc="upper left")
plt.tight_layout(); plt.show()

ot = Table(title="gaze vs concept-copier head overlap")
ot.add_column("metric"); ot.add_column("value", justify="right")
for k, v in [("gaze heads", len(gset)), ("concept heads", len(cset)),
             ("intersection", len(inter)), ("union", len(union)), ("Jaccard", f"{jacc:.3f}")]:
    ot.add_row(k, str(v))
ot.add_row("overlap heads", ", ".join(f"L{l}H{h}" for l, h in sorted(inter)) or "—")
rich_print(ot)

## 4. Do concept-copier heads show *gaze* behavior?

Feed the concept-copier heads into our **gaze visualization** (per-step attention over the condition
spans, `track_conditions`) and eyeball whether they track the current condition like the gaze heads do.
The actual gaze heads are shown below as reference.

In [ ]:
from src.attention import show_interactive_attention
from src.goals.multiline_prompts import build_example
from src.functional import generate_with_patch

demo_ex = build_example(mt=mt, rng=random.Random(42), target_slot=2, n_conditions=5,
                        mode="chat", naming="letters")
demo_prompt = demo_ex["full_prompt"]
demo_gen = generate_with_patch(mt, inputs=demo_prompt, max_new_tokens=128,
                               do_sample=False, remove_prefix=True, n_gen_per_prompt=1)[0]
rich_print(f"[white]{escape(demo_prompt)}[/white][sea_green1]{escape(demo_gen)}[/sea_green1]")

ai = get_attention_matrices(demo_prompt + demo_gen, mt, value_weighted=True)
toks = ai.tokenized_prompt
start = next((i - 1 for i, t in enumerate(toks) if "user" in t.replace("＜", "<")), 0)

# TODO #4: the gaze view, fed the CONCEPT-copier heads
show_interactive_attention(attn_info=ai, selected_heads=sorted(concept_heads), start_from=max(0, start),
                           agg="mean", normalize="row_max", cmap="Purples", track_conditions=demo_ex["spec"],
                           height=None, title=f"CONCEPT-copier heads ({len(concept_heads)}) in the gaze view")

# reference: the actual gaze heads on the same attention
show_interactive_attention(attn_info=ai, selected_heads=sorted(gaze_heads), start_from=max(0, start),
                           agg="mean", normalize="row_max", cmap="Greens", track_conditions=demo_ex["spec"],
                           height=None, title=f"GAZE heads ({len(gaze_heads)}) — reference")